[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/evals/openai_agents_cookbook.ipynb)


<center>
<p style="text-align:center">
<img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
<br>
<br>
<a href="https://arize.com/docs/phoenix/">文档</a>
|
<a href="https://github.com/Arize-ai/phoenix">GitHub</a>
|
<a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
</p>
</center>
<h1 align="center">追踪与评估 OpenAI Agent</h1>

本指南介绍如何使用 Phoenix 创建和评估 Agent，以改进性能。步骤包括：

* 使用 OpenAI Agents SDK 创建 Agent
* 追踪 Agent 活动
* 创建用于性能基准测试的数据集
* 运行实验，使用 LLM 裁判评估 Agent 性能
* 学习如何评估生产环境中的追踪

# 初始设置


### 安装库

In [ ]:
%pip install -q "arize-phoenix>=8.0.0" openinference-instrumentation-openai-agents openinference-instrumentation-openai --upgrade
%pip install -q openai nest_asyncio openai-agents

### 配置依赖与密钥

接下来连接 Phoenix。以下代码连接 Phoenix Cloud 实例。你也可以选择[连接自托管 Phoenix 实例](https://arize.com/docs/phoenix/deployment)。

In [ ]:
import os
from getpass import getpass

import nest_asyncio

nest_asyncio.apply()

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 请输入 Phoenix 收集器端点：")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint


if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥：")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥：")
os.environ["OPENAI_API_KEY"] = openai_api_key

### 配置追踪

In [ ]:
from phoenix.otel import register

# 配置追踪
tracer_provider = register(
    project_name="openai-agents-cookbook",
    auto_instrument=True,
)

# 使用 OpenAI SDK 创建第一个 Agent

这里配置了一个能解决数学问题的基本 Agent。

我们有一个用于求解数学方程的函数工具，以及能够使用它的 Agent。

使用 `Runner` 类运行 Agent 并获取最终输出。

In [ ]:
from agents import Runner, function_tool


@function_tool
def solve_equation(equation: str) -> str:
    """使用 Python 计算数学表达式，不要自行心算。

    参数：
       equation：传入 Python eval() 的表达式字符串。
    """
    return str(eval(equation))

In [ ]:
from agents import Agent

agent = Agent(
    name="Math Solver",
    instructions="使用 Python 计算数学问题并返回结果",
    tools=[solve_equation],
)

In [ ]:
result = await Runner.run(agent, "what is 15 + 28?")

# 运行结果对象
print(result)

# 获取最终输出
print(result.final_output)

# 获取生成最终输出时记录的完整消息列表
print(result.to_input_list())

基本 Agent 已创建，下面评估它是否回答正确！

# 评估 Agent

Agent 可能因为多种原因出错。
1. 工具调用准确性：Agent 是否选择了正确工具并使用正确参数？
2. 工具调用结果：工具是否返回正确结果？
3. Agent 目标准确性：Agent 是否完成既定目标并得到正确结果？

我们将配置一个简单评估器来检查 Agent 回答是否正确。有关不同类型的 Agent 评估，请参阅[此处](https://docs.arize.com/arize/llm-evaluation-and-annotations/how-does-evaluation-work/agent-evaluation)。

定义任务函数、评估器和数据集，配置评估。

In [ ]:
import asyncio


# 这是任务函数，接收问题并返回最终输出及生成输出时记录的消息。
async def solve_math_problem(dataset_row: dict):
    result = await Runner.run(agent, dataset_row.get("question"))
    return {
        "final_output": result.final_output,
        "messages": result.to_input_list(),
    }


dataset_row = {"question": "What is 15 + 28?"}

result = asyncio.run(solve_math_problem(dataset_row))
print(result)

接下来创建评估器。

In [ ]:
from phoenix.evals import (
    ClassificationEvaluator,
    bind_evaluator,
)
from phoenix.evals.llm import LLM

# 用于评估数学解答的模板
MATH_EVAL_TEMPLATE = """
你正在评估数学问题是否被正确解决。

[BEGIN DATA]
************
[Question]: {question}
************
[Response]: {response}
[END DATA]

判断数学问题的答案是否正确。先自行求出正确答案，再与给定回复比较。
注意同一答案可能有不同表达方式，
例如 "43" 与 "The answer is 43"，或 "5.0" 与 "5"。

答案必须是单个词，"correct" 或 "incorrect"。
"""

llm = LLM(provider="openai", model="gpt-4.1")

math_eval = ClassificationEvaluator(
    name="math eval",
    prompt_template=MATH_EVAL_TEMPLATE,
    llm=llm,
    choices={"correct": 1.0, "incorrect": 0.0},
)

correctness_eval = bind_evaluator(
    evaluator=math_eval,
    input_mapping={
        "question": "input.question",
        "response": "output.final_output",
    },
)

# 创建合成问题数据集

使用下面的模板生成一个包含 25 个问题的 DataFrame，用于测试数学解题 Agent。

In [ ]:
MATH_GEN_TEMPLATE = """
你是生成多样化数学问题的助手，用于测试数学解题 Agent。
问题应包括：

基本运算：简单加、减、乘、除。
复杂算术：包含多种运算和括号，并遵循运算顺序。
幂与根：涉及幂、平方根和其他 n 次根。
百分比：计算某数的百分比或百分比变化。
分数：分数加、减、乘、除。
代数：可代入特定值求值的简单代数式。
数列：求数字序列的和、积或平均数。
应用题：将文字题转为数学表达式。

生成问题时不要包含答案。

以列表回复，每行一道题，行首不要编号。
生成 25 道不同的数学问题，确保不重复。
"""

In [ ]:
import nest_asyncio
import pandas as pd

nest_asyncio.apply()
pd.set_option("display.max_colwidth", 500)

# 初始化模型
model = LLM(provider="openai", model="gpt-4o")

# 生成数学问题
resp = model.generate_text(prompt=MATH_GEN_TEMPLATE)

# 创建 DataFrame
split_response = resp.strip().split("\n")
math_problems_df = pd.DataFrame(split_response, columns=["question"])
print(math_problems_df.head())

现在用这个数据集运行 Agent！

# 开发阶段的实验

开发期间，实验可以通过评估暴露 Agent 的失败，帮助快速迭代。部署前可使用数据集测试，完善提示词、逻辑与工具用法。

本节在上面定义的数据集上运行 Agent，并使用 LLM 裁判评估正确性。

## 创建实验

使用上面生成的问题数据集，可以通过实验功能跟踪 Agent 的模型、提示词和参数变化。

创建此数据集并上传至平台。

In [ ]:
import uuid

from phoenix.client import Client

unique_id = uuid.uuid4()

dataset_name = "math-questions-" + str(uuid.uuid4())[:5]

# 将数据集上传到 Phoenix
dataset = Client().datasets.create_dataset(
    dataframe=math_problems_df,
    input_keys=["question"],
    name=f"math-questions-{unique_id}",
)
print(dataset)

In [ ]:
from phoenix.client.experiments import run_experiment

initial_experiment = run_experiment(
    dataset,
    task=solve_math_problem,
    evaluators=[correctness_eval],
    experiment_description="Solve Math Problems",
    experiment_name=f"solve-math-questions-{str(uuid.uuid4())[:5]}",
)

## 在 Phoenix 中查看追踪

![结果](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/experiment_in_development.gif)

# 生产环境中的评估

生产环境中的评估提供实时洞察，反映 Agent 在用户数据上的表现。

本节模拟线上生产环境，展示如何实时收集追踪、模型输出和评估结果。

也可以获取已完成的生产运行追踪，对其进行批量评估，再将评估结果记录到 Phoenix。

In [ ]:
!pip install openinference-instrumentation

In [ ]:
from opentelemetry.trace import StatusCode, format_span_id

导入必要的库后，配置追踪器对象，以便创建 span 并追踪任务函数。

In [ ]:
tracer = tracer_provider.get_tracer(__name__)

接下来更新正确性评估器，使其同时返回标签和解释，便于在追踪期间捕获元数据。

我们也修改任务函数，增加在 Phoenix 中生成结构化 span 的 `with` 语句。这些 span 捕获输入值、输出值及评估结果等关键信息。

In [ ]:
# 生产中直接调用评估器，捕获完整 Score 对象。
def production_correctness_eval(input, output):
    result = math_eval.evaluate(
        eval_input={"question": input["question"], "response": output["final_output"]}
    )
    return result[0] if result else None

In [ ]:
# 这是修改后的任务函数。
async def solve_math_problem(dataset_row: dict):
    with tracer.start_as_current_span(name="agent", openinference_span_kind="agent") as agent_span:
        question = dataset_row.get("question")
        agent_span.set_input(question)
        agent_span.set_status(StatusCode.OK)

        result = await Runner.run(agent, question)
        agent_span.set_output(result.final_output)

        task_result = {
            "final_output": result.final_output,
            "messages": result.to_input_list(),
        }

        # 正确性评估 span
        with tracer.start_as_current_span(
            "correctness-evaluator",
            openinference_span_kind="evaluator",
        ) as eval_span:
            evaluation_result = production_correctness_eval(dataset_row, task_result)
            eval_span.set_attribute("eval.label", evaluation_result.label)
            eval_span.set_attribute("eval.explanation", evaluation_result.explanation)

        # 记录评估
        span_id = format_span_id(eval_span.get_span_context().span_id)
        eval_data = {
            "span_id": span_id,
            "label": evaluation_result.label,
            "score": evaluation_result.score,
            "explanation": evaluation_result.explanation,
        }
        df = pd.DataFrame([eval_data])
        from phoenix.client import AsyncClient

        px_client = AsyncClient()
        await px_client.spans.log_span_annotations_dataframe(
            dataframe=df,
            annotation_name="correctness",
            annotator_kind="LLM",
        )

    return task_result


dataset_row = {"question": "What is 15 + 28?"}

result = asyncio.run(solve_math_problem(dataset_row))
print(result)

最后运行实验，模拟生产环境中的追踪。

In [ ]:
from phoenix.client.experiments import run_experiment

initial_experiment = run_experiment(
    dataset,
    task=solve_math_problem,
    experiment_description="Solve Math Problems",
    experiment_name=f"solve-math-questions-{str(uuid.uuid4())[:5]}",
)

## 追踪生成后，在 Phoenix 中查看追踪与评估器结果

![结果 2](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/evaluating_openai_agents.gif)